In [11]:
conda install pytables

Retrieving notices: done
Channels:
 - conda-forge
 - nodefaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.9.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /opt/conda/envs/python3

  added / updated specs:
    - pytables


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  file:///conda-forge
    certifi-2026.2.25          |     pyhd8ed1ab_0         148 KB  file:///conda-forge
    openssl-3.6.2              |       h35e630c_0         3.0 MB  file:///conda-forge
    py-cpuinfo-9.0.0           |     pyhd8ed1ab_1          25 KB  file:///conda-forge
    pytables-3.10.2            | py312hefc0c3f_10         1.6 MB  file:///conda-forge
    -----------

In [ ]:
import glob
import os
import numpy as np
import rasterio as rio
from rasterio.plot import show
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Any, NamedTuple, Optional
import tqdm
import cmocean as cm
from concurrent.futures import ProcessPoolExecutor, as_completed

# import tables
import vaex as vx

In [ ]:
# Import masks and example file
with rio.open('/home/jovyan/work/AVOCA/GEM/Development/Data/Masks/Landmask.tif') as landmask_file:
    landmask = landmask_file.read()
    landmask_1d = landmask.ravel()
    is_land = (landmask_1d == 1)

with rio.open('/home/jovyan/work/AVOCA/GEM/Development/Data/Masks/Icemask.tif') as icemask_file:
    icemask = icemask_file.read()
    icemask_1d = icemask.ravel()
    is_ice = (icemask_1d == 1)

lst_example = rio.open('/home/jovyan/work/AVOCA/GEM/Development/Data/BatchExport_LST/2000/GEMLST_MODIS_20000101.tif')


# Specify output directories
OUTPUT_DIR = "/home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/Output_tables"
OUTPUT_TRAIN_DIR = os.path.join(OUTPUT_DIR, "Training")
OUTPUT_TRAIN_ICE_DIR = os.path.join(OUTPUT_TRAIN_DIR, "Ice")
OUTPUT_TRAIN_LAND_DIR = os.path.join(OUTPUT_TRAIN_DIR, "Land")

OUTPUT_TEST_DIR = os.path.join(OUTPUT_DIR, "Testing")
OUTPUT_TEST_ICE_DIR = os.path.join(OUTPUT_TEST_DIR, "Ice")
OUTPUT_TEST_LAND_DIR = os.path.join(OUTPUT_TEST_DIR, "Land")


# Set number of cores for parallel processing
NUM_CORES = 10

In [ ]:
# SATLST Data: 
satfolder = "/home/jovyan/work/AVOCA/GEM/Development/Data/BatchExport_LST/YYYY/GEMLST_MODIS_yyyymmdd.tif"
era5folder = "/home/jovyan/work/AVOCA/GEM/Development/Data/ERA5/GL1000m_reproj"

# List all files in the folder and sort them
imfiles = sorted(Path(era5folder).glob("*.tif"))
# print(f"Found files {imfiles} in the folder.")

# markers (adjust if you need case-insensitive match or different substrings)
start_marker = "t2m_1000m_2000_d001.tif" # OBS D11
end_marker = "t2m_1000m_2001_d365.tif"   # OBS LEAP YEARS

# find first index containing the start marker and last index containing the end marker
start_idx = next((i for i, p in enumerate(imfiles) if start_marker in p.name), None)
end_idx = next((i for i, p in enumerate(imfiles) if end_marker in p.name), None)

if start_idx is None:
    raise FileNotFoundError(f"No file containing '{start_marker}' found under {era5folder}")
if end_idx is None:
    raise FileNotFoundError(f"No file containing '{end_marker}' found under {era5folder}")
if end_idx < start_idx:
    raise ValueError(f"End file '{end_marker}' appears before start file '{start_marker}' in sorted file order")

# slice inclusive range
imfiles = imfiles[start_idx:end_idx + 1]



In [ ]:
with ProcessPoolExecutor(
    max_workers=NUM_WORKERS,
    initializer=init_worker,
    initargs=(CALIBRATION_CSV,),
) as executor:
    futures = [executor.submit(process_single_carra_file, carra_fp, OUTPUT_TRAIN_DIR, OUTPUT_TEST_DIR) for carra_fp in carra_files]

    success = 0
    failed = 0


    with tqdm.tqdm(enumerate(imfiles), total=len(imfiles), desc="Processing") as pbar:
        for idx, imfile in pbar:
            

        ### EXTRACT DATE AND FIND SAT IMAGE ###
            # Find the corresponding SATLST image based on the ERA5 filename
            yearstring = imfile.stem.split("_")[2]
            year = pd.to_datetime(yearstring, format="%Y")
            doy = imfile.stem.split("_")[3][1:]
            date = year + pd.to_timedelta(int(doy) - 1, unit="D")
            date = date.strftime("%Y-%m-%d")
            datestring = date.replace("-", "")
            imagepath = satfolder.replace('YYYY', yearstring).replace('yyyymmdd', datestring)

            # print(f'ERA5 file {imfile.stem} matches SATLST file {Path(imagepath).stem} for date {date}.')

            # Open image files and read data
            with rio.open(imagepath) as sat:
                sat_lst = sat.read(1)
                sat_qa = sat.read(2)
            # Set values to NaN where qa band is 0 (no sat obs)
                sat_lst = np.where(sat_qa == 0, np.nan, sat_lst)

            with rio.open(imfile) as era5_file:
                era5_t2m = era5_file.read(1)


        ### ARRANGE ARRAYS ####
            # Transform images into a 1D array (numpy ravel)
            era5_1d = era5_t2m.ravel()
            satlst_1d = sat_lst.ravel()
            # print(f'ERA5 1D shape: {era5_1d.shape}, SATLST 1D shape: {satlst_1d.shape}, Landmask 1D shape: {landmask_1d.shape}')


        ### MASKING AND NA HANDLING ###
            na_mask = ~np.isnan(era5_1d) & ~np.isnan(satlst_1d)
            na_mask_ice = na_mask & is_ice
            na_mask_land = na_mask & is_land

            #era5_1d_nona = era5_1d[na_mask]
            #satlst_1d_nona = satlst_1d[na_mask]
            era5_1d_nona_ice = era5_1d[na_mask_ice]
            satlst_1d_nona_ice = satlst_1d[na_mask_ice]
            era5_1d_nona_land = era5_1d[na_mask_land]
            satlst_1d_nona_land = satlst_1d[na_mask_land]


        ######## 3. Precompute Landmask Masks
        # If landmask_1d is static, precompute landmask_1d == 0 and landmask_1d == 1 outside the loop (if this is inside a loop).


        ### Build dataframes and split into training and testing (testing = 0.3)
            df_ice = pd.DataFrame({'date': [date] * len(era5_1d_nona_ice), 'era5': era5_1d_nona_ice, 'satlst': satlst_1d_nona_ice})        
            df_land = pd.DataFrame({'date': [date] * len(era5_1d_nona_land), 'era5': era5_1d_nona_land, 'satlst': satlst_1d_nona_land})

            ice_train, ice_test = train_test_split(df_ice, test_size=0.3, random_state=42)
            land_train, land_test = train_test_split(df_land, test_size=0.3, random_state=42)


            
        ### SAVE FILES ### 
            vx_ice_train = vx.from_pandas(ice_train, copy_index=False)
            vx_ice_test = vx.from_pandas(ice_test, copy_index=False)
            vx_land_train = vx.from_pandas(land_train, copy_index=False)
            vx_land_test = vx.from_pandas(land_test, copy_index=False)

            # filenames
            ice_train_filename = os.path.join(OUTPUT_TRAIN_ICE_DIR, f"comparison_train_ice_{date}.h5")
            ice_test_filename = os.path.join(OUTPUT_TEST_ICE_DIR, f"comparison_test_ice_{date}.h5")
            land_train_filename = os.path.join(OUTPUT_TRAIN_LAND_DIR, f"comparison_train_land_{date}.h5")
            land_test_filename = os.path.join(OUTPUT_TEST_LAND_DIR, f"comparison_test_land_{date}.h5")

            vx_ice_train.export_hdf5(ice_train_filename, mode="w")
            vx_ice_test.export_hdf5(ice_test_filename, mode="w")
            vx_land_train.export_hdf5(land_train_filename, mode="w")
            vx_land_test.export_hdf5(land_test_filename, mode="w")


        ### PRINTS###
            # Every round
            # print(f'File {imfile} processed, corresponding to date {date}. Total valid points: {len(era5_1d_nona)}, Ice valid points: {len(era5_1d_nona_ice)}, Land valid points: {len(era5_1d_nona_land)}')

            pbar.set_postfix(last_index=idx, last_file=imfile)

        # Confirm loop completion
        print('ALL FILES DONE') 


In [ ]:
# Read HDF5 files to confirm they were written correctly
#total_df = pd.read_hdf(path_results_total, key="data")
#ice_df = pd.read_hdf(path_results_ice, key="data")
land_df = pd.read_hdf(path_results_land, key="data")

#print(total_df.head(), "\n", total_df.shape[0])
#print(ice_df.head(), "\n", ice_df.shape[0])
print(land_df.head(), "\n", land_df.shape[0])

## From Shunans scrips

In [ ]:
# df = vx.open("/data_3/shunan_2/KU/GEMLST/Output_tables/Comparison_Land.h5")
df.head()

# Shuffle the dataset
df = df.shuffle(random_state=42)

# Split the dataset into training (2/3) and testing data (1/3)
df_train, df_test = df.ml.train_test_split(test_size=0.3)


#,table
0,"(0, [b'2000-02-24'], [-23.018024], [-27.60041519])"
1,"(1, [b'2000-02-24'], [-23.173948], [-27.85788187])"
2,"(2, [b'2000-02-24'], [-23.18668], [-27.57012499])"
3,"(3, [b'2000-02-24'], [-23.179543], [-27.84273678])"
4,"(4, [b'2000-02-24'], [-23.174736], [-27.90331717])"
5,"(5, [b'2000-02-24'], [-23.201519], [-27.22178772])"
6,"(6, [b'2000-02-24'], [-23.188578], [-27.5246897])"
7,"(7, [b'2000-02-24'], [-23.1807], [-27.76701128])"
8,"(8, [b'2000-02-24'], [-23.723785], [-28.20621915])"
9,"(9, [b'2000-02-24'], [-23.679298], [-28.20621915])"


In [5]:
# Shuffle the dataset
df = df.shuffle(random_state=42)

# Split the dataset into training (2/3) and testing data (1/3)
df_train, df_test = df.ml.train_test_split(test_size=0.3)

df.head()

/data/shunan/conda-envs/bigdata/lib/python3.10/site-packages/vaex/ml/__init__.py:28: UserWarning: Make sure the DataFrame is shuffled
  warnings.warn('Make sure the DataFrame is shuffled')


#,table
0,"""(159863, [b'2006-10-07'], [-13.072565], [-13.61..."
1,"""(159231, [b'2014-07-02'], [1.1384315], [2.92355..."
2,"""(10228, [b'2016-11-02'], [-14.6316395], [-17.13..."
3,"""(198398, [b'2002-04-11'], [-14.215612], [-17.38..."
4,"(39006, [b'2005-07-27'], [2.414449], [7.1280967])"
5,"""(6680, [b'2019-10-29'], [-24.555302], [-24.6713..."
6,"""(8055, [b'2004-02-06'], [-42.07934], [-29.48715..."
7,"""(67351, [b'2002-10-11'], [-13.2390785], [-14.26..."
8,"""(33320, [b'2003-01-03'], [-29.451227], [-23.247..."
9,"""(165094, [b'2001-11-28'], [-15.786383], [-7.689..."


In [ ]:
CALIBRATION_CAP = 0.83

class RegressionResult(NamedTuple):
    slope: float
    intercept: float
    rvalue: float
    pvalue: float


In [ ]:
def linregress_vaex(df: Any, x: str, y: str, selection: Optional[str] = None) -> RegressionResult:
    """Compute linear-regression summary from Vaex aggregations (out-of-core)."""
    base_selection = f"isfinite({x}) & isfinite({y})"
    if selection:
        selection = f"({base_selection}) & ({selection})"
    else:
        selection = base_selection

    n = float(df.count(selection=selection))
    if n < 3:
        raise ValueError("Need at least 3 finite points to compute regression statistics")

    mean_x = float(df.mean(x, selection=selection))
    mean_y = float(df.mean(y, selection=selection))
    var_x = float(df.var(x, selection=selection))
    var_y = float(df.var(y, selection=selection))
    mean_xy = float(df.mean(df[x] * df[y], selection=selection))
    cov_xy = mean_xy - mean_x * mean_y

    if var_x <= 0 or var_y <= 0:
        raise ValueError("Variance is zero; regression is undefined")

    slope = cov_xy / var_x
    intercept = mean_y - slope * mean_x
    rvalue = cov_xy / np.sqrt(var_x * var_y)

    # Numerical guard to keep r within [-1, 1] for p-value computation.
    rvalue = float(np.clip(rvalue, -1.0, 1.0))
    if abs(rvalue) == 1.0:
        pvalue = 0.0
    else:
        t_stat = rvalue * np.sqrt((n - 2.0) / (1.0 - rvalue**2))
        pvalue = float(2.0 * stats.t.sf(abs(t_stat), df=n - 2.0))
    # r2 = df.ml.metrics.r2_score(df[y], df[x], selection=selection)
    return RegressionResult(
        slope=float(slope),
        intercept=float(intercept),
        rvalue=float(rvalue),
        pvalue=float(pvalue),
    )


## Old parts of the script

In [ ]:
### LINEAR REGRESSION ### 
    # print(f'Image:{date}')

    slope, intercept, r_value, p_value, std_err = linregress(era5_1d_nona, satlst_1d_nona)
    # print(f'Total\tslope: {round(slope, 3)}, intercept: {round(intercept, 3)}, r_value: {round(r_value, 2)}, p_value: {round(p_value,3)}, std_err: {round(std_err, 3)}')

    slope_i, intercept_i, r_value_i, p_value_i, std_err_i = linregress(era5_1d_nona_ice, satlst_1d_nona_ice)
    # print(f'Ice\tslope: {round(slope_i, 3)}, intercept: {round(intercept_i, 3)}, r_value: {round(r_value_i, 2)}, p_value: {round(p_value_i,3)}, std_err: {round(std_err_i, 3)}')

    slope_l, intercept_l, r_value_l, p_value_l, std_err_l = linregress(era5_1d_nona_land, satlst_1d_nona_land)
    # print(f'Land\tslope: {round(slope_l, 3)}, intercept: {round(intercept_l, 3)}, r_value: {round(r_value_l, 2)}, p_value: {round(p_value_l,3)}, std_err: {round(std_err_l, 3)}\n')


### PLOT (for test purposes) 

    fig, axes = plt.subplots(1, 3, figsize=(12, 6))
    xlims = (era5_1d_nona.min(), era5_1d_nona.max())
    ylims = (satlst_1d_nona.min(), satlst_1d_nona.max())
    
    # Plot 1 total
    ax = axes[0]
    hexbin = ax.hexbin(era5_1d_nona, satlst_1d_nona,
                    gridsize=50, cmap=cmocean.cm.haline, bins='log', mincnt=1)
    
    sns.regplot(ax=ax, x=era5_1d_nona, y=satlst_1d_nona, scatter=False, color='red')
    ax.plot([-50, 30], [-50, 30], 'k--', alpha=0.5, label='1:1 line')
    ax.set_title('Total')
    ax.set_aspect('equal')
    ax.set_xlim([-50, 30])
    ax.set_ylim([-50, 30])
    ax.set_xlabel('ERA5 T2M (°C)')
    ax.set_ylabel('SATLST (°C)')
    cb = fig.colorbar(hexbin, ax=ax)
    cb.set_label('log(Count+1)')

    # Add statistics text box for training data
    stats_text = f'N = {era5_1d_nona.shape[0]}\n'
    stats_text += f'R² = {r_value**2:.3f}\n'
    stats_text += f'Slope = {slope:.3f}\n'
    stats_text += f'Intercept = {intercept:.3f}\n'
   
    ax.text(0.50, 0.35, stats_text, transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))


    # Plot 2 Ice
    ax = axes[1]
    hexbin = ax.hexbin(era5_1d_nona_ice, satlst_1d_nona_ice,
                    gridsize=50, cmap=cmocean.cm.haline, bins='log', mincnt=1)
    
    sns.regplot(ax=ax, x=era5_1d_nona_ice, y=satlst_1d_nona_ice, scatter=False, color='red')
    ax.plot([-50, 30], [-50, 30], 'k--', alpha=0.5, label='1:1 line')
    ax.set_title('Ice')
    ax.set_aspect('equal')
    ax.set_xlim([-50, 30])
    ax.set_ylim([-50, 30])
    ax.set_xlabel('ERA5 T2M (°C)')
    ax.set_ylabel('SATLST (°C)')
    cb = fig.colorbar(hexbin, ax=ax)
    cb.set_label('log(Count+1)')

    # Add statistics text box for training data
    stats_text = f'N = {era5_1d_nona_ice.shape[0]}\n'
    stats_text += f'R² = {r_value_i**2:.3f}\n'
    stats_text += f'Slope = {slope_i:.3f}\n'
    stats_text += f'Intercept = {intercept_i:.3f}\n'
   
    ax.text(0.50, 0.35, stats_text, transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    # Plot 3 Land
    ax = axes[2]
    hexbin = ax.hexbin(era5_1d_nona_land, satlst_1d_nona_land,
                    gridsize=50, cmap=cmocean.cm.haline, bins='log', mincnt=1)
    
    sns.regplot(ax=ax, x=era5_1d_nona_land, y=satlst_1d_nona_land, scatter=False, color='red')
    ax.plot([-50, 30], [-50, 30], alpha=0.5, label='1:1 line')
    ax.set_title('Land')
    ax.set_aspect('equal')
    ax.set_xlim([-50, 30])
    ax.set_ylim([-50, 30])
    ax.set_xlabel('ERA5 T2M (°C)')
    ax.set_ylabel('SATLST (°C)')
    cb = fig.colorbar(hexbin, ax=ax)
    cb.set_label('log(Count+1)')

    # Add statistics text box for training data
    stats_text = f'N = {era5_1d_nona_land.shape[0]}\n'
    stats_text += f'R² = {r_value_l**2:.3f}\n'
    stats_text += f'Slope = {slope_l:.3f}\n'
    stats_text += f'Intercept = {intercept_l:.3f}\n'
   
    ax.text(0.50, 0.35, stats_text, transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    fig.suptitle(f'ERA5 vs SATLST on {date}', fontsize=16)
    plt.tight_layout()
    plt.show()